In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
from src import operators as op


In [3]:

rng = np.random.default_rng(0)
pop = op.init_population(5, 3, (-5.12, 5.12), rng)
print(pop)
print(pop.shape)                                   # should be (5, 3)
print(pop.min() >= -5.12, pop.max() <= 5.12)       # should be True True


def sphere(x):
    return np.sum(x ** 2)

fitness = np.array([sphere(x) for x in pop])
print(fitness)

[[ 1.40248768 -2.35738405 -4.70043111]
 [-4.95075701  3.20788725  4.22661711]
 [ 1.09195034  2.35004478  0.44671991]
 [ 4.45514162  3.23434039 -5.09195776]
 [ 3.65981979 -4.77608371  2.35167177]]
(5, 3)
True True
[29.61828392 52.6648278   6.91462472 56.23727845 41.73561663]


In [4]:
parents = op.tournament_selection(pop, fitness, 5, 3, rng)
print(parents)
print([sphere(p) for p in parents])

[[ 1.40248768 -2.35738405 -4.70043111]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.09195034  2.35004478  0.44671991]
 [ 1.40248768 -2.35738405 -4.70043111]]
[29.6182839174636, 6.9146247228092435, 6.9146247228092435, 6.9146247228092435, 29.6182839174636]


In [5]:
p1 = np.array([2.0, 0.0, -1.0])
p2 = np.array([4.0, 1.0, -1.0])
c1, c2 = op.blx_alpha_crossover(p1, p2, (-5.12, 5.12), rng)
print(c1, c2)

[ 3.58875805  0.73077022 -1.        ] [ 4.98883974  1.46167068 -1.        ]


In [6]:
c1, c2 = op.sbx_crossover(p1, p2, (-5.12, 5.12), rng)
print(c1, c2)
print((c1 + c2) / 2, (p1 + p2) / 2)   # should match

[ 1.97737357 -0.01500312 -1.        ] [ 4.02262643  1.01500312 -1.        ]
[ 3.   0.5 -1. ] [ 3.   0.5 -1. ]


In [7]:
kids = op.crossover(parents, 0.9, "sbx", (-5.12, 5.12), rng)
print(kids.shape)                                         # same as parents.shape
kids_none = op.crossover(parents, 0.0, "sbx", (-5.12, 5.12), rng)
print(np.array_equal(kids_none, parents))                 # rate 0 -> True
# op.crossover(parents, 0.9, "abc", (-5.12, 5.12), rng)     # should raise a clear error

(5, 3)
True


In [8]:
m = op.gaussian_mutation(pop, 0.3, (-5.12, 5.12), rng)
print((m != pop).mean())                  # roughly 0.3 (fraction of genes changed)
print(m.min() >= -5.12, m.max() <= 5.12)  # True True

0.3333333333333333
True True


In [9]:
m = op.polynomial_mutation(pop, 1.0, (-5.12, 5.12), rng)   # mutate every gene
print(np.round(m - pop, 2))       # changes: small, both + and -
print((m == 5.12).mean())         # should be ~0, NOT most genes stuck at the edge

[[ 0.93 -0.69  0.29]
 [ 0.9   1.26 -1.58]
 [ 0.61  1.48  1.13]
 [-0.57  1.32  0.71]
 [ 0.49 -0.02 -0.37]]
0.0


In [10]:
new = op.init_population(5, 3, (-5.12, 5.12), rng)
new_fit = np.array([sphere(x) for x in new])
p, f = op.apply_elitism(pop, fitness, new, new_fit, 2)
print(fitness.min(), f.min())    # f.min() can never be worse than fitness.min()
print(np.allclose(f, [sphere(x) for x in p]))   # True: scores match the solutions

6.9146247228092435 6.9146247228092435
True


In [11]:
from src import ga
res = ga.run(sphere, (-5.12, 5.12), 10, {"seed": 0})
print(res)

{'best_f': 3.6182331576006155e-10, 'evals': 100000, 'history': [14.325347328513885, 13.05121251183364, 11.523947875941627, 10.398803070651972, 9.507394022046281, 8.709395847256774, 7.923464679144484, 6.47937723242203, 6.081611529334162, 5.742485057838953, 5.154263081632937, 4.823260548361371, 4.493203456197927, 3.9221529733362224, 3.3137352965572915, 2.970246934701278, 2.2375547868279066, 2.08878644103811, 2.08878644103811, 1.7465200875574842, 1.2521818143026524, 1.2521818143026524, 1.2521818143026524, 1.0492938810097625, 0.7925643105883828, 0.7435676319089786, 0.6126582974672696, 0.50146313624486, 0.4972497976794725, 0.48390594061917663, 0.281838076225319, 0.2617331287595925, 0.23330074925902397, 0.23096500596119318, 0.227270426279373, 0.18512187597294605, 0.18512187597294605, 0.18512187597294605, 0.1763387381987053, 0.15543243237583962, 0.15299353698160184, 0.14122987061280767, 0.11746505860064041, 0.11746505860064041, 0.10527506793164128, 0.09486357736298995, 0.09486357736298995, 0.

In [12]:
res = ga.run(sphere, (-5.12, 5.12), 10, {"seed": 0, "max_evals": 20000})
print(res["best_f"], res["evals"])    # best_f should be close to 0, evals 20000
print(res["history"][:5], res["history"][-1])   # numbers going down

3.313731139030706e-06 20000
[14.325347328513885, 13.05121251183364, 11.523947875941627, 10.398803070651972, 9.507394022046281] 3.313731139030706e-06


abit harder tests

In [13]:
def rastrigin(x):
    return 10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x))

def ackley(x):
    n = len(x)
    return (-20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n))
            - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.e)


In [18]:

for f, b in [(sphere, (-5.12, 5.12)), (rastrigin, (-5.12, 5.12)), (ackley, (-32.768, 32.768))]:
    r = ga.run(f, b, 10, {"seed": 0, "max_evals": 100000})
    print(f.__name__, r["history"][0], "->", r["best_f"])

sphere 14.325347328513885 -> 3.6182331576006155e-10
rastrigin 96.43005529669438 -> 2.664201588231663e-07
ackley 17.10235126221705 -> 0.00014307790480261318


In [17]:
res = ga.run(sphere, (-5.12, 5.12), 10, {"seed": 0, "max_evals": 20000})
print(res["best_x"])                         # 10 numbers, all close to 0
print(sphere(res["best_x"]), res["best_f"])  # these two should be equal

[ 3.28486324e-05 -3.16320313e-04  6.86529626e-04 -3.77459228e-04
 -3.93370581e-04  4.65972493e-04  1.13081141e-03  5.04634099e-04
  2.85014229e-04  7.82496814e-04]
3.313731139030706e-06 3.313731139030706e-06


In [16]:
for bad in [dict(dim=0), dict(bounds=(5, -5)), dict(pop=7), dict(cx=1.5)]:
    try:
        ga.run(sphere, bad.get("bounds", (-5.12, 5.12)), bad.get("dim", 10),
               {"pop_size": bad.get("pop", 100), "crossover_rate": bad.get("cx", 0.9)})
    except ValueError as e:
        print("OK:", e)

OK: dim must be at least 1, got 0
OK: bounds must have low < high, got (5, -5)
OK: pop_size must be an even number >= 2, got 7
OK: crossover_rate must be between 0 and 1, got 1.5


In [19]:
from src import ma
rng = np.random.default_rng(0)
x = np.array([3.0, -2.0, 1.0])
x2, f2, used = ma.solis_wets(sphere, x, sphere(x), (-5.12, 5.12), rng, max_steps=200, budget=1000)
print(sphere(x), "->", f2)      # should drop a lot (from 14 toward 0)
print(used <= 1000)             # True
print(np.isclose(sphere(x2), f2))   # True: score matches solution

14.0 -> 2.833055355005266e-17
True
True


In [20]:
_, _, used = ma.solis_wets(sphere, x, sphere(x), (-5.12, 5.12), rng, max_steps=200, budget=10)
print(used)    # must be 10 or less

9


In [24]:
from src import ga, ma
for f, b in [(sphere, (-5.12, 5.12)), (rastrigin, (-5.12, 5.12)), (ackley, (-32.768, 32.768))]:
    g = ga.run(f, b, 10, {"seed": 0, "max_evals": 100000})
    m = ma.run(f, b, 10, {"seed": 0, "max_evals": 100000})
    print(f"{f.__name__:10s} GA {g['best_f']:.2e} ({g['evals']})   MA {m['best_f']:.2e} ({m['evals']})")

sphere     GA 3.62e-10 (100000)   MA 4.36e-14 (99903)
rastrigin  GA 2.66e-07 (100000)   MA 1.51e-11 (99998)
ackley     GA 1.43e-04 (100000)   MA 2.34e-06 (99999)


ma better than ga

before tuning for focused local search

sphere     GA 3.62e-10 (100000)   MA 1.23e-09 (99986)
rastrigin  GA 2.66e-07 (100000)   MA 2.40e-07 (99940)
ackley     GA 1.43e-04 (100000)   MA 9.77e-05 (99991)

after:
sphere     GA 3.62e-10 (100000)   MA 4.36e-14 (99903)
rastrigin  GA 2.66e-07 (100000)   MA 1.51e-11 (99998)
ackley     GA 1.43e-04 (100000)   MA 2.34e-06 (99999)

In [25]:
from src import ga, ma
for f, b in [(sphere, (-5.12, 5.12)), (rastrigin, (-5.12, 5.12)), (ackley, (-32.768, 32.768))]:
    g = ga.run(f, b, 30, {"seed": 0, "max_evals": 300000})
    m = ma.run(f, b, 30, {"seed": 0, "max_evals": 300000})
    print(f"{f.__name__:10s} GA {g['best_f']:.2e} ({g['evals']})   MA {m['best_f']:.2e} ({m['evals']})")

sphere     GA 6.17e-10 (300000)   MA 6.45e-13 (299999)
rastrigin  GA 1.24e-06 (300000)   MA 1.83e-10 (299903)
ackley     GA 1.64e-04 (300000)   MA 3.90e-06 (299925)


ma worse on all 3 functions insane evil eye

before:

sphere     GA 6.17e-10 (300000)   MA 2.11e-09 (299941)
rastrigin  GA 1.24e-06 (300000)   MA 2.60e-06 (300000)
ackley     GA 1.64e-04 (300000)   MA 2.87e-04 (299921)

after:
sphere     GA 6.17e-10 (300000)   MA 6.45e-13 (299999)
rastrigin  GA 1.24e-06 (300000)   MA 1.83e-10 (299903)
ackley     GA 1.64e-04 (300000)   MA 3.90e-06 (299925)

now MA beats GA in all the 6 cases above

In [23]:
#with ls_rho(local search rho)

f, b = ackley, (-32.768, 32.768)
g = ga.run(f, b, 30, {"seed": 0, "max_evals": 300000})
print("GA", f"{g['best_f']:.2e}")
for s in [dict(ls_rho=0.01),
          dict(ls_rho=0.001),
          dict(ls_rho=0.001, ls_every=10, ls_top_k=1, ls_steps=50)]:
    m = ma.run(f, b, 30, {"seed": 0, "max_evals": 300000, **s})
    print(s, f"{m['best_f']:.2e}")

GA 1.64e-04
{'ls_rho': 0.01} 2.07e-04
{'ls_rho': 0.001} 1.39e-04
{'ls_rho': 0.001, 'ls_every': 10, 'ls_top_k': 1, 'ls_steps': 50} 3.90e-06
